# DART JAX denoiser TPU checks

This notebook runs the JAX denoiser and diffusion `q_sample` checks on Kaggle TPU v5e-8.

Upload/mount these inputs:

- the DART repo snapshot containing `jax_dart/`
- a Torch-exported denoiser parity snapshot `.npz` made locally with `--mode torch-export --torch-device cpu`
- optionally, the TPU shard dataset with `metadata.json` and `shards/` for path sanity checks

Use an MLP or Transformer Torch-exported parity snapshot depending on the model you want to validate.

In [3]:
# Edit these paths for your Kaggle mount names.
REPO_IN = "/kaggle/input/datasets/markuskamsties/darttpu"
# Keep REPO in /kaggle/working. Kaggle /kaggle/input mounts are read-only.
REPO = "/kaggle/working/DART"

# CPU Torch-exported snapshot from your local DART env.
SNAPSHOT = "/kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz"

# Optional shard root for path checks only. Leave empty if not mounted in this notebook.
ROOT = ""
OUT_DIR = "/kaggle/working/parity_outputs"
OUT_COMPARE = f"{OUT_DIR}/denoiser_compare_real_batch_random_mlp_cpu.npz"

import os
os.environ.update({
    "REPO_IN": REPO_IN,
    "REPO": REPO,
    "SNAPSHOT": SNAPSHOT,
    "ROOT": ROOT,
    "OUT_DIR": OUT_DIR,
    "OUT_COMPARE": OUT_COMPARE,
})

print("REPO_IN:", REPO_IN)
print("SNAPSHOT:", SNAPSHOT)
print("ROOT:", ROOT)
print("OUT_COMPARE:", OUT_COMPARE)

REPO_IN: /kaggle/input/datasets/markuskamsties/darttpu
SNAPSHOT: /kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz
ROOT: 
OUT_COMPARE: /kaggle/working/parity_outputs/denoiser_compare_real_batch_random_mlp_cpu.npz


## Copy repo to writable working space

In [4]:
import os
if not os.environ["REPO"].startswith("/kaggle/working/"):
    raise RuntimeError(f"REPO must be under /kaggle/working, got {os.environ['REPO']}")

!rm -rf "$REPO"
!cp -r "$REPO_IN" "$REPO"
%cd /kaggle/working/DART
!ls -lah jax_dart/kaggle
!ls -lah jax_dart/models | grep denoiser || true
!mkdir -p "$OUT_DIR"

/kaggle/working/DART
total 28K
drwxr-xr-x 2 root root 4.0K May  1 00:08 .
drwxr-xr-x 8 root root 4.0K May  1 00:08 ..
-rw-r--r-- 1 root root   61 May  1 00:08 __init__.py
-rw-r--r-- 1 root root 5.1K May  1 00:08 denoiser_mlp_run.py
-rw-r--r-- 1 root root 5.7K May  1 00:08 vae_partial_run.py
-rw-r--r-- 1 root root 8.7K May  1 00:08 mld_denoiser.py


## Optional dependency refresh

In [5]:
# Uncomment only if the Kaggle image does not already expose the expected TPU JAX stack.
!pip install -q -U "jax[tpu]" flax optax orbax-checkpoint -f https://storage.googleapis.com/jax-releases/libtpu_releases.html


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


## TPU/path check

In [6]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --check-only \
  --snapshot "$SNAPSHOT" \
  --root "$ROOT"

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
snapshot: /kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz (ok)
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777594154.836500     217 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:226
jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)

## JAX-only parity compare

In [7]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --task parity \
  --mode jax-compare \
  --snapshot "$SNAPSHOT" \
  --atol 1e-4 \
  --rtol 1e-4 \
  --fail-on-mismatch \
  --out-npz "$OUT_COMPARE"

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
snapshot: /kaggle/input/datasets/markuskamsties/snapshot/denoiser_torch_real_batch_random_mlp_cpu.npz (ok)
out_npz: /kaggle/working/parity_outputs/denoiser_compare_real_batch_random_mlp_cpu.npz
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777594204.899915     973 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:226
jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_c

## Denoiser Transformer TPU smoke

In [8]:
!python jax_dart/kaggle/denoiser_mlp_run.py \
  --platform tpu \
  --require-tpu \
  --expect-devices 8 \
  --task smoke \
  --model-type transformer \
  --batch-size 8 \
  --h-dim 512 \
  --ff-size 1024 \
  --num-layers 8 \
  --num-heads 4 \
  --clip-dim 512 \
  --history-shape "2 276" \
  --noise-shape "1 256" \
  --n-blocks 2

repo_root: /kaggle/working/DART
JAX_PLATFORMS: tpu
/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:91: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1777594251.988101    2073 common_lib.cc:638] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:226
jax_backend: tpu
jax_process_index: 0
jax_process_count: 1
jax_local_device_count: 8
jax_devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=

## Inspect compare output

In [9]:
import os
import numpy as np

with np.load(os.environ["OUT_COMPARE"]) as data:
    print("arrays:", sorted(data.files))
    for name in ["x_t", "denoised"]:
        torch_value = data[f"torch_{name}"]
        jax_value = data[f"jax_{name}"]
        diff = np.abs(torch_value - jax_value)
        print(name, "shape", torch_value.shape, "max_abs", diff.max(), "mean_abs", diff.mean())

arrays: ['config_json', 'history', 'jax_denoised', 'jax_x_t', 'noise', 'state::embed_timestep.sequence_pos_encoder.pe', 'state::embed_timestep.time_embed.0.bias', 'state::embed_timestep.time_embed.0.weight', 'state::embed_timestep.time_embed.2.bias', 'state::embed_timestep.time_embed.2.weight', 'state::input_project.bias', 'state::input_project.weight', 'state::mlp.layers.0.layers.0.bias', 'state::mlp.layers.0.layers.0.weight', 'state::mlp.layers.0.layers.1.bias', 'state::mlp.layers.0.layers.1.weight', 'state::mlp.layers.1.layers.0.bias', 'state::mlp.layers.1.layers.0.weight', 'state::mlp.layers.1.layers.1.bias', 'state::mlp.layers.1.layers.1.weight', 'state::mlp.out_fc.bias', 'state::mlp.out_fc.weight', 'state::sequence_pos_encoder.pe', 'text_embedding', 'timesteps', 'torch_denoised', 'torch_x_t', 'x_start']
x_t shape (32, 1, 256) max_abs 0.0 mean_abs 0.0
denoised shape (32, 1, 256) max_abs 5.364418e-07 mean_abs 8.2625235e-08


## Package outputs

In [ ]:
!cd /kaggle/working && zip -r dart_jax_denoiser_mlp_tpu_outputs.zip parity_outputs